# Fase 3: las medidas de forma de la célula

Hasta ahora el modelo solo mira la foto. Pero el dataset trae algo más: el contorno del borde de
la célula y el del núcleo, dibujados a mano por especialistas. De ahí se pueden sacar medidas
concretas, las mismas que mira un citotecnólogo en el microscopio: qué tan grande es el núcleo,
cuánto espacio ocupa dentro de la célula, qué tan redondo es, qué tan oscuro.

La pregunta de esta fase es simple: **¿darle esas medidas al modelo lo mejora?**

Para responderla hay que comparar bien. El error típico es entrenar un modelo grande con medidas
y compararlo contra otro modelo distinto sin medidas: ahí cambian dos cosas a la vez y no se sabe
cuál causó la diferencia. Nosotros vamos a usar **exactamente la misma red, el mismo entrenamiento
y los mismos pliegues**, y lo único que cambia es si recibe las medidas o no.

Tres modelos a comparar:

1. **Solo medidas**, sin mirar la foto (un modelo clásico de árboles).
2. **Solo foto**, la red sola.
3. **Foto más medidas**, la red con las medidas pegadas al final.

## Preparativos

Lo mismo que en la fase 2: GPU, Drive y el dataset.

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
print("Modelo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [ ]:
import pandas as pd, numpy as np, kagglehub

DEST = "/content/drive/MyDrive/sipakmed"
df = pd.read_csv(f"{DEST}/indice_celulas.csv").reset_index(drop=True)

path = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")

# Corregir las rutas guardadas, que apuntan a la carpeta de la sesion anterior
base_viejo = df["img"].iloc[0].split("/im_")[0]
for col in ["img", "cyt", "nuc"]:
    df[col] = df[col].str.replace(base_viejo, path, regex=False)

print("Filas:", len(df), "| imágenes encontradas:", df["img"].apply(os.path.exists).all())

Volvemos a armar los mismos pliegues por lámina de siempre. Misma semilla, mismo resultado, así
todo lo de esta fase se compara contra la fase 2 sin trampas.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

SEMILLA = 42
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
df["fold_lamina"] = -1
for k, (_, val) in enumerate(sgkf.split(df, df["clase"], groups=df["lamina"])):
    df.loc[df.index[val], "fold_lamina"] = k

CLASES = sorted(df["clase"].unique())
IDX = {c: i for i, c in enumerate(CLASES)}
Y = df["clase"].map(IDX).values
print(df.groupby("fold_lamina").size())

## Sacar las medidas

Por cada célula calculamos unas 25 medidas a partir de los dos contornos y de la foto. Van
agrupadas así:

- **Tamaño:** área y perímetro del núcleo y de la célula, y el diámetro equivalente.
- **Proporción núcleo/célula:** cuánto del espacio de la célula ocupa el núcleo. Es la medida más
  usada en el diagnóstico real, porque en las células anormales el núcleo crece.
- **Forma:** qué tan redondo es cada contorno, qué tan alargado, y si tiene entrantes o es liso.
- **Posición:** si el núcleo está centrado o corrido hacia un lado.
- **Color y textura:** qué tan oscuro es el núcleo comparado con el resto de la célula, y qué tan
  parejo o granulado se ve por dentro.
- **Tamaño original del recorte**, que es la información que se pierde cuando escalamos todo a
  224x224.

Tarda unos minutos porque abre las 4049 imágenes y sus 8098 contornos.

In [ ]:
import cv2
from skimage.feature import graycomatrix, graycoprops

def medidas_contorno(cnt):
    """Devuelve area, perimetro y descriptores de forma de un contorno."""
    area = cv2.contourArea(cnt)
    per = cv2.arcLength(cnt, True)
    hull = cv2.convexHull(cnt)
    area_hull = cv2.contourArea(hull)

    # Circularidad: 1 = circulo perfecto, mas bajo = mas irregular
    circ = 4 * np.pi * area / (per ** 2) if per > 0 else np.nan
    # Solidez: que tan "lleno" es el contorno respecto a su envoltura convexa
    sol = area / area_hull if area_hull > 0 else np.nan
    diam = np.sqrt(4 * area / np.pi)

    # Elongacion: hace falta un minimo de 5 puntos para ajustar una elipse
    if len(cnt) >= 5:
        (_, _), (eje1, eje2), _ = cv2.fitEllipse(cnt)
        mayor, menor = max(eje1, eje2), min(eje1, eje2)
        alarg = menor / mayor if mayor > 0 else np.nan
        exc = np.sqrt(1 - (menor / mayor) ** 2) if mayor > 0 else np.nan
    else:
        alarg = exc = np.nan

    M = cv2.moments(cnt)
    cx = M["m10"] / M["m00"] if M["m00"] > 0 else np.nan
    cy = M["m01"] / M["m00"] if M["m00"] > 0 else np.nan
    return area, per, circ, sol, diam, alarg, exc, cx, cy


def leer_contorno(ruta):
    pts = np.loadtxt(ruta, delimiter=",")
    return pts.reshape(-1, 2).astype(np.int32).reshape(-1, 1, 2)


def caracteristicas(fila):
    img = cv2.imread(fila["img"])
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    alto, ancho = gris.shape

    cyt = leer_contorno(fila["cyt"])
    nuc = leer_contorno(fila["nuc"])

    a_c, p_c, ci_c, so_c, d_c, al_c, ex_c, cx_c, cy_c = medidas_contorno(cyt)
    a_n, p_n, ci_n, so_n, d_n, al_n, ex_n, cx_n, cy_n = medidas_contorno(nuc)

    # Mascaras para medir color dentro de cada region
    m_cyt = np.zeros(gris.shape, np.uint8); cv2.fillPoly(m_cyt, [cyt.reshape(-1, 2)], 1)
    m_nuc = np.zeros(gris.shape, np.uint8); cv2.fillPoly(m_nuc, [nuc.reshape(-1, 2)], 1)
    m_cit = np.clip(m_cyt.astype(int) - m_nuc.astype(int), 0, 1).astype(np.uint8)  # citoplasma sin nucleo

    px_n = gris[m_nuc == 1]
    px_c = gris[m_cit == 1]
    int_n = px_n.mean() if px_n.size else np.nan
    int_c = px_c.mean() if px_c.size else np.nan
    std_n = px_n.std() if px_n.size else np.nan

    # Textura del nucleo: que tan granulado se ve por dentro
    x, y, w, h = cv2.boundingRect(nuc)
    recorte = gris[y:y+h, x:x+w]
    if recorte.size > 16:
        q = (recorte // 8).astype(np.uint8)          # bajamos a 32 niveles de gris
        glcm = graycomatrix(q, [1], [0], levels=32, symmetric=True, normed=True)
        contraste = graycoprops(glcm, "contrast")[0, 0]
        homog = graycoprops(glcm, "homogeneity")[0, 0]
        energia = graycoprops(glcm, "energy")[0, 0]
    else:
        contraste = homog = energia = np.nan

    desplaz = np.hypot(cx_n - cx_c, cy_n - cy_c) / np.sqrt(a_c) if a_c > 0 else np.nan

    return {
        "nuc_area": a_n, "cyt_area": a_c, "nc_area": a_n / a_c if a_c > 0 else np.nan,
        "nuc_per": p_n, "cyt_per": p_c, "nc_per": p_n / p_c if p_c > 0 else np.nan,
        "nuc_circ": ci_n, "cyt_circ": ci_c,
        "nuc_sol": so_n, "cyt_sol": so_c,
        "nuc_diam": d_n, "cyt_diam": d_c,
        "nuc_alarg": al_n, "cyt_alarg": al_c,
        "nuc_exc": ex_n, "cyt_exc": ex_c,
        "desplaz_nucleo": desplaz,
        "int_nuc": int_n, "int_cit": int_c,
        "contraste_nuc_cit": int_c - int_n if not (np.isnan(int_c) or np.isnan(int_n)) else np.nan,
        "std_nuc": std_n,
        "glcm_contraste": contraste, "glcm_homog": homog, "glcm_energia": energia,
        "ancho_orig": ancho, "alto_orig": alto,
    }


print("Calculando medidas...")
lista = []
for i, fila in df.iterrows():
    lista.append(caracteristicas(fila))
    if i % 400 == 0:
        print(f"  {i}/{len(df)}", end="\r")

F = pd.DataFrame(lista)
COLS_F = list(F.columns)
print(f"Listo: {len(COLS_F)} medidas por célula")
print("Valores que no se pudieron calcular:", F.isna().sum().sum())
F.describe().T[["mean", "std", "min", "max"]].round(3)

Antes de meterlas al modelo, vale mirar si estas medidas por sí solas ya separan las clases.
Si la proporción núcleo/célula sale claramente distinta entre tipos, es buena señal.

In [ ]:
import matplotlib.pyplot as plt

tmp = F.copy(); tmp["clase"] = df["clase"].values
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, var, titulo in zip(axes,
                           ["nc_area", "int_nuc", "nuc_circ"],
                           ["Proporción núcleo/célula", "Oscuridad del núcleo", "Redondez del núcleo"]):
    datos = [tmp[tmp.clase == c][var].dropna() for c in CLASES]
    ax.boxplot(datos, labels=[c[:12] for c in CLASES], showfliers=False)
    ax.set_title(titulo); ax.tick_params(axis="x", rotation=45, labelsize=8)
plt.tight_layout(); plt.show()

F.to_csv(f"{DEST}/medidas_fase3.csv", index=False)
print("Medidas guardadas")

## Modelo 1: solo las medidas

Un modelo de árboles clásico, sin mirar la foto en ningún momento. Usa los mismos cinco pliegues
por lámina.

Este modelo tiene dos gracias. Una, es rapidísimo. Dos, es completamente explicable: se puede
preguntar qué medida pesó más en cada decisión, cosa que con la red no se puede.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score

pred_medidas = np.full(len(df), -1)

for k in range(5):
    tr = np.where(df["fold_lamina"].values != k)[0]
    te = np.where(df["fold_lamina"].values == k)[0]

    # Este modelo maneja los valores faltantes por dentro, no hay que rellenarlos
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                       random_state=SEMILLA)
    m.fit(F.iloc[tr], Y[tr])
    pred_medidas[te] = m.predict(F.iloc[te])

print("Solo medidas:")
print(f"  macro F1 {f1_score(Y, pred_medidas, average='macro'):.4f}"
      f" | MCC {matthews_corrcoef(Y, pred_medidas):.4f}"
      f" | acc {accuracy_score(Y, pred_medidas):.4f}")

## Cargar las imágenes

Igual que en la fase 2: se decodifican una sola vez y quedan en memoria, para que las épocas no
dependan del disco.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

print("Cargando imágenes...")
CACHE = np.zeros((len(df), 224, 224, 3), dtype=np.uint8)
for i, ruta in enumerate(df["img"]):
    CACHE[i] = np.array(Image.open(ruta).convert("RGB").resize((224, 224), Image.BILINEAR))
    if i % 500 == 0:
        print(f"  {i}/{len(df)}", end="\r")
print(f"Listo: {CACHE.nbytes/1e6:.0f} MB")

MEDIA = [0.485, 0.456, 0.406]
DESV  = [0.229, 0.224, 0.225]

tf_train = v2.Compose([
    v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(), v2.RandomRotation(20),
    v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEDIA, DESV),
])
tf_eval = v2.Compose([v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEDIA, DESV)])


class CelulasF(Dataset):
    """Entrega la foto y el vector de medidas ya escalado."""
    def __init__(self, posiciones, feats, tf):
        self.pos = np.asarray(posiciones)
        self.feats = feats.astype(np.float32)
        self.tf = tf

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, i):
        p = self.pos[i]
        img = torch.from_numpy(CACHE[p]).permute(2, 0, 1)
        return self.tf(img), torch.from_numpy(self.feats[i]), Y[p]

## La red, con y sin medidas

Una sola clase que sirve para las dos condiciones. EfficientNet-B0 saca 1280 números que resumen
la foto; si `usar_medidas` está activado, se le pegan las medidas a ese vector antes de la capa
final. Si no, la capa final recibe solo los 1280.

**Este es el punto de la fase:** las dos condiciones comparten la misma red, la misma cabeza, el
mismo entrenamiento y los mismos pliegues. Cambia una sola cosa.

In [ ]:
import torch.nn as nn, random, time
from torchvision import models

EPOCAS = 12
LOTE = 64
LR = 3e-4
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def fijar_semilla(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


class RedFusion(nn.Module):
    def __init__(self, n_medidas, n_clases, usar_medidas):
        super().__init__()
        self.usar_medidas = usar_medidas
        base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        base.classifier = nn.Identity()          # nos quedamos con los 1280 numeros
        self.base = base
        entrada = 1280 + (n_medidas if usar_medidas else 0)
        self.cabeza = nn.Sequential(
            nn.Linear(entrada, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, n_clases),
        )

    def forward(self, img, med):
        z = self.base(img)
        if self.usar_medidas:
            z = torch.cat([z, med], dim=1)
        return self.cabeza(z)

Un detalle que parece menor y no lo es: **las medidas se escalan usando solo el grupo de
entrenamiento**. El promedio y la desviación se calculan con los cuatro pliegues de práctica, y
esos mismos valores se aplican al pliegue de examen.

Si se calcularan sobre todas las células a la vez, el modelo estaría recibiendo información del
examen antes de rendirlo. Es una fuga sutil, fácil de cometer sin darse cuenta, y arruina la
comparación igual que la del reparto al azar de la fase 2. Lo mismo vale para rellenar los valores
que faltan: la mediana se saca del grupo de práctica y nunca de la clase real de la célula.

In [ ]:
def escalar(F, tr, te):
    """Escala las medidas con las estadisticas del grupo de entrenamiento unicamente."""
    mu = F.iloc[tr].mean()
    sd = F.iloc[tr].std().replace(0, 1)
    med_tr = F.iloc[tr].median()            # para rellenar faltantes, tambien solo del train

    a = ((F.iloc[tr].fillna(med_tr) - mu) / sd).values
    b = ((F.iloc[te].fillna(med_tr) - mu) / sd).values
    return np.nan_to_num(a), np.nan_to_num(b)


def entrenar(k, usar_medidas):
    fijar_semilla(SEMILLA)

    tr = np.where(df["fold_lamina"].values != k)[0]
    te = np.where(df["fold_lamina"].values == k)[0]
    f_tr, f_te = escalar(F, tr, te)

    dl_tr = DataLoader(CelulasF(tr, f_tr, tf_train), batch_size=LOTE, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)
    dl_te = DataLoader(CelulasF(te, f_te, tf_eval), batch_size=128, shuffle=False,
                       num_workers=2, pin_memory=True)

    modelo = RedFusion(F.shape[1], len(CLASES), usar_medidas).to(DEV)
    opt = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCAS)
    criterio = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    for ep in range(EPOCAS):
        modelo.train()
        for img, med, y in dl_tr:
            img, med, y = img.to(DEV), med.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterio(modelo(img, med), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()

    modelo.eval()
    preds = []
    with torch.no_grad():
        for img, med, y in dl_te:
            with torch.amp.autocast("cuda"):
                salida = modelo(img.to(DEV), med.to(DEV))
            preds.append(salida.argmax(1).cpu().numpy())

    del modelo; torch.cuda.empty_cache()
    return te, np.concatenate(preds)

## Entrenar las dos condiciones

Cinco pliegues por condición, diez entrenamientos. Con las imágenes en memoria debería ir
bastante más rápido que la fase 2.

In [ ]:
pred_solo_img = np.full(len(df), -1)
pred_fusion   = np.full(len(df), -1)
filas = []

for nombre, usar, destino in [("SOLO IMAGEN", False, pred_solo_img),
                              ("IMAGEN + MEDIDAS", True, pred_fusion)]:
    print(f"\n=== {nombre} ===")
    for k in range(5):
        t0 = time.time()
        te, pred = entrenar(k, usar)
        destino[te] = pred

        f1 = f1_score(Y[te], pred, average="macro")
        filas.append({"modelo": nombre, "pliegue": k, "macro_f1": f1,
                      "mcc": matthews_corrcoef(Y[te], pred),
                      "accuracy": accuracy_score(Y[te], pred)})
        print(f"  pliegue {k}: F1 {f1:.4f}  ({time.time()-t0:.0f}s)")

res3 = pd.DataFrame(filas)
res3.groupby("modelo")[["macro_f1", "mcc", "accuracy"]].agg(["mean", "std"]).round(4)

## Los tres modelos, lado a lado

In [ ]:
resumen = pd.DataFrame([
    {"modelo": "Solo medidas",
     "macro_f1": f1_score(Y, pred_medidas, average="macro"),
     "mcc": matthews_corrcoef(Y, pred_medidas),
     "accuracy": accuracy_score(Y, pred_medidas)},
    {"modelo": "Solo imagen",
     "macro_f1": f1_score(Y, pred_solo_img, average="macro"),
     "mcc": matthews_corrcoef(Y, pred_solo_img),
     "accuracy": accuracy_score(Y, pred_solo_img)},
    {"modelo": "Imagen + medidas",
     "macro_f1": f1_score(Y, pred_fusion, average="macro"),
     "mcc": matthews_corrcoef(Y, pred_fusion),
     "accuracy": accuracy_score(Y, pred_fusion)},
]).round(4)
print(resumen.to_string(index=False))

ganancia = resumen.loc[2, "macro_f1"] - resumen.loc[1, "macro_f1"]
print(f"\nGanancia de sumar las medidas: {ganancia:+.4f}")

## ¿La diferencia es real?

Otra vez la prueba de McNemar, comparando célula por célula el modelo con medidas contra el mismo
modelo sin ellas. Si la ganancia es chica, esta prueba dice si vale la pena o es ruido.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

ok_img = (pred_solo_img == Y)
ok_fus = (pred_fusion == Y)
tabla = [[int((ok_img & ok_fus).sum()),  int((ok_img & ~ok_fus).sum())],
         [int((~ok_img & ok_fus).sum()), int((~ok_img & ~ok_fus).sum())]]

print("Aciertan las dos:", tabla[0][0])
print("Solo acierta la imagen sola:", tabla[0][1])
print("Solo acierta con medidas:", tabla[1][0])
print("Fallan las dos:", tabla[1][1])

r = mcnemar(tabla, exact=False, correction=True)
print(f"\np = {r.pvalue:.2e}")
print("La mejora es real" if r.pvalue < 0.05 else "No hay diferencia clara")

## ¿En qué clase ayudan las medidas?

La hipótesis era que las medidas ayudan sobre todo en la clase que más se confunde. Acá se ve
clase por clase.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

r_img = classification_report(Y, pred_solo_img, target_names=CLASES, output_dict=True)
r_fus = classification_report(Y, pred_fusion, target_names=CLASES, output_dict=True)

comp = pd.DataFrame({
    "F1 solo imagen": [r_img[c]["f1-score"] for c in CLASES],
    "F1 con medidas": [r_fus[c]["f1-score"] for c in CLASES],
}, index=CLASES).round(3)
comp["diferencia"] = (comp["F1 con medidas"] - comp["F1 solo imagen"]).round(3)
print(comp.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (titulo, pred) in zip(axes, [("Solo imagen", pred_solo_img),
                                     ("Imagen + medidas", pred_fusion)]):
    cm = confusion_matrix(Y, pred, normalize="true")
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASES))); ax.set_yticks(range(len(CLASES)))
    ax.set_xticklabels(CLASES, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(CLASES, fontsize=8)
    ax.set_title(titulo); ax.set_xlabel("predicho"); ax.set_ylabel("real")
    for i in range(len(CLASES)):
        for j in range(len(CLASES)):
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
plt.tight_layout(); plt.show()

## Qué medidas pesan más

Ventaja del modelo de árboles: se le puede preguntar en qué se fijó. Esto es material directo para
la discusión, porque se puede contrastar con lo que dice la literatura médica.

In [ ]:
from sklearn.inspection import permutation_importance

tr = np.where(df["fold_lamina"].values != 0)[0]
te = np.where(df["fold_lamina"].values == 0)[0]
m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, random_state=SEMILLA)
m.fit(F.iloc[tr], Y[tr])

imp = permutation_importance(m, F.iloc[te], Y[te], n_repeats=5,
                             random_state=SEMILLA, scoring="f1_macro")
orden = np.argsort(imp.importances_mean)[::-1][:12]

plt.figure(figsize=(8, 5))
plt.barh([COLS_F[i] for i in orden][::-1], imp.importances_mean[orden][::-1])
plt.xlabel("Cuánto empeora el modelo si se desordena esa medida")
plt.title("Medidas más importantes")
plt.tight_layout(); plt.show()

## Guardar

In [ ]:
res3.to_csv(f"{DEST}/resultados_fase3.csv", index=False)
salida = df[["stem", "clase", "lamina", "fold_lamina"]].copy()
salida["pred_medidas"] = pred_medidas
salida["pred_solo_img"] = pred_solo_img
salida["pred_fusion"] = pred_fusion
salida.to_csv(f"{DEST}/predicciones_fase3.csv", index=False)
print("Guardado en", DEST)

## Resumen de la fase 3

Lo que queda establecido:

- Un modelo que solo usa medidas de forma, sin ver la foto, ya llega a cierto nivel. Sirve de
  referencia y además es explicable.
- La comparación entre la red con y sin medidas es limpia: misma arquitectura, mismo
  entrenamiento, mismos pliegues, una sola variable cambiada. Si hay mejora, se sabe de dónde viene.
- El escalado de las medidas se hace solo con el grupo de entrenamiento, evitando la fuga sutil
  que arruinaría la comparación.

Si la ganancia resulta pequeña o nula, hay que reportarlo tal cual: significaría que la red ya
estaba captando por su cuenta la información de forma que le estamos dando explícita.

**Lo que sigue (fase 4):** todas estas medidas salen de contornos dibujados a mano por un
especialista, y en un laboratorio real eso no existe. La fase 4 entrena un modelo que dibuje los
contornos solo, recalcula las medidas a partir de ahí, y compara los tres escenarios: sin medidas,
con medidas hechas a mano y con medidas automáticas.